In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

False

In [3]:
MODEL="llama3.2"

In [4]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaEmbeddings

llm = ChatOllama(model=MODEL)
embeddings=OllamaEmbeddings(model="nomic-embed-text")

parser = StrOutputParser()
chain= llm | parser

In [5]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader("WEEK-1.pdf")

pages=loader.load_and_split()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14036\1078237039.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [6]:
from langchain_core.prompts import PromptTemplate
template = """
Answer the question based on the context below. If you can't 
answer the question, reply "I don't know".

Context: {context}

Question: {question}
"""
prompt=PromptTemplate.from_template(template)
prompt.format(context="Here is some context",question="Here is the question")

'\nAnswer the question based on the context below. If you can\'t \nanswer the question, reply "I don\'t know".\n\nContext: Here is some context\n\nQuestion: Here is the question\n'

In [7]:
chain =  prompt | llm | parser

In [9]:
chain.invoke(
    {
        "context":"Today is Tuesday at 9 PM",
        "question":"What is the time today?"
    }
)

'The current time is 9 PM.'

In [8]:
from langchain_community.vectorstores import DocArrayInMemorySearch
vectorstore=DocArrayInMemorySearch.from_documents(
    pages,
    embedding=embeddings
)

In [9]:
retriever= vectorstore.as_retriever()
retriever.invoke("Ethical Hacking")

[Document(metadata={'producer': '3-Heights™ PDF Toolbox API 6.12.0.6 (http://www.pdf-tools.com)', 'creator': 'PowerPoint', 'creationdate': '2019-07-22T15:42:08+00:00', 'moddate': '2022-01-17T06:45:46+00:00', 'source': 'WEEK-1.pdf', 'total_pages': 79, 'page': 1, 'page_label': '2'}, page_content='q\u202f\tWhat\tis\tethical\thacking?\t\nq\u202f\tPenetra1on\ttes1ng\t\nq\u202f\tRole\tof\tthe\tethical\thacker'),
 Document(metadata={'producer': '3-Heights™ PDF Toolbox API 6.12.0.6 (http://www.pdf-tools.com)', 'creator': 'PowerPoint', 'creationdate': '2019-07-22T15:42:08+00:00', 'moddate': '2022-01-17T06:45:46+00:00', 'source': 'WEEK-1.pdf', 'total_pages': 79, 'page': 2, 'page_label': '3'}, page_content="What\tis\tEthical\tHacking?\t\n•\u202fIt\trefers\tto\tthe\tact\tof\tloca1ng\tweaknesses\tand\tvulnerabili1es\tof\tcomputer\tand\t\ninforma1on\tsystems\tby\treplica1ng\tthe\tintent\tand\tac1ons\tof\tmalicious\thackers.\t\n•\u202fIt\tis\talso\tknown\tas\tpenetra'on\ttes'ng,\tintrusion\ttes'ng\to

In [12]:
from operator import itemgetter
chain=(
    {
        "context":itemgetter("question") | retriever,
        "question":itemgetter("question")
    }
    | prompt
    | llm
    | parser
)
chain.invoke({"question":"what is ethical hacking?"})

'Ethical hacking refers to the act of locating weaknesses and vulnerabilities of computer systems and information systems by replicating the intent and actions of malicious hackers, also known as penetration testing or red teaming. It is also referred to as white-hat hacking.'

In [10]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader("WEEK-3.pdf")

page=loader.load_and_split()

vectorstore1=DocArrayInMemorySearch.from_documents(
    page,
    embedding=embeddings
)

retriever1=vectorstore1.as_retriever()


In [13]:
chain1 = (
    {
        "context": itemgetter("question") | retriever1,
        "question": itemgetter("question")
    }
    | prompt
    | llm
    | parser
)

In [ ]:

questions=[
    "Which routing table flag  indicates that a route is specific for a single host rather than an entire network?",
    "In the OSFP routing protocol,which type of packet is used to determine whether a neighbouring router is active?",
    "What type of address should be used when a packet needs t be delivered to every host within a specific network?",
    "What is the address used to represent the default route in routing table in IPv4?",
    "How many bits are used for hop limit in IPv6 base header?"
]
for question in questions:
    print(f"Question: {question}")
    print(chain1.invoke({'question':question}))
    print()

Question: Which routing table flag  indicates that a route is specific for a single host rather than an entire network?
The answer to the question based on the provided context is:

H. Host-specific address

This flag is mentioned in page 5 of the PDF document, in section d) Host-specific routing: "Can specify the address of a host."

Question: In the OSFP routing protocol,which type of packet is used to determine whether a neighbouring router is active?
The answer to the question is "Hello" packet. According to the document, in the OSPF routing protocol, the "Hello" packet is used to check if a neighboring router is up.

Question: What type of address should be used when a packet needs t be delivered to every host within a specific network?
The answer based on the provided context is "Multicast". According to the page_content text, "• Multicast: Refers to a set of computers, possibly at different locations. Packet delivered to every member of the set."

Question: What is the address u

In [14]:
quiz_prompt = PromptTemplate.from_template("""
You are an AI study assistant.

Generate exactly 5 multiple-choice questions using ONLY the provided context.

Return ONLY a valid JSON array in this format:

[
    {{
        "question": "Question text",
        "options": {{
            "A": "First option",
            "B": "Second option",
            "C": "Third option",
            "D": "Fourth option"
        }},
        "answer": "A"
    }}
]

Rules:
- Generate exactly 5 questions.
- Each question must have exactly 4 options: A, B, C, D.
- Each option must contain a complete answer, not just a letter.
- There must be exactly one correct answer.
- "answer" must contain only A, B, C, or D.
- The correct answer must match one of the options.
- Use ONLY information from the context.
- Do not add explanations or text outside the JSON.
- Do not create a question whose options are about a different topic than the question.

Before generating each question, verify that:
- The question is directly supported by the context.
- The question does not assume relationships or terminology that are not stated in the context.
- The correct answer is explicitly supported by the context.
- The question is not circular (the answer should not simply repeat the question).
- Do not ask about synonyms, "another names", or relationships unless they are explicitly stated in the context.
- For questions about protocols, distinguish between a protocol, a protocol class, and a routing method.
- All four options must be plausible answers to the specific question, but only one may be supported as correct by the context.

Context:
{context}
""")

In [15]:
import json
context = retriever1.invoke("computer networks routing")
context_text = "\n\n".join(
    doc.page_content for doc in context
)

chain_quiz=quiz_prompt|llm|parser
quiz=json.loads(chain_quiz.invoke({"context":context_text}))
print(quiz)

[{'question': 'What type of routing is based on the destination network address?', 'options': {'A': 'Host-specific routing', 'B': 'Network-specific routing', 'C': 'Default routing', 'D': 'Next-hop routing'}, 'answer': 'B'}, {'question': 'What is the primary purpose of Border Gateway Protocol (BGP)?', 'options': {'A': 'To provide default routing', 'B': 'To use interior routing protocols', 'C': 'To establish exterior routing protocols', 'D': 'Border Gateway Protocol (BGP)'}, 'answer': 'C'}, {'question': 'What is the characteristic of a Static Routing Table?', 'options': {'A': 'Updated periodically depending on network condition', 'B': 'Does not change with time', 'C': 'Uses protocols like RIP, OSPF, BGP, etc.', 'D': 'Contains information inserted manually'}, 'answer': 'B'}, {'question': 'Which routing method is based on the next hop?', 'options': {'A': 'Network-specific routing', 'B': 'Host-specific routing', 'C': 'Next-hop routing', 'D': 'Default routing'}, 'answer': 'C'}, {'question': 

In [ ]:
print(quiz[0]["question"])
print(quiz[0]["options"]["A"])
print(quiz[0]["answer"])

What type of routing is based on the destination network address?
Host-specific routing
B


In [16]:
score=0
for i,question in enumerate(quiz):
    print(f"\nQuestion {i + 1}: {question['question']}")
    for option,text in question['options'].items():
        print(f"{option}.{text}")
    answer=input("Your Answer: ").isupper()

    if answer==question["answer"]:
        print("That's Correct!")
        score+=1
    else:
        print("Wrong.")
print(f"\nYour score: {score}/{len(quiz)}")


Question 1: What type of routing is based on the destination network address?
A.Host-specific routing
B.Network-specific routing
C.Default routing
D.Next-hop routing
Wrong.

Question 2: What is the primary purpose of Border Gateway Protocol (BGP)?
A.To provide default routing
B.To use interior routing protocols
C.To establish exterior routing protocols
D.Border Gateway Protocol (BGP)
Wrong.

Question 3: What is the characteristic of a Static Routing Table?
A.Updated periodically depending on network condition
B.Does not change with time
C.Uses protocols like RIP, OSPF, BGP, etc.
D.Contains information inserted manually
Wrong.

Question 4: Which routing method is based on the next hop?
A.Network-specific routing
B.Host-specific routing
C.Next-hop routing
D.Default routing
Wrong.

Question 5: What is the primary difference between Interior and Exterior routing protocols?
A.Interior protocols are used for external routes
B.Exterior protocols are used for internal routes
C.Interior protoc